# Student Performance Prediction Using Linear Regression and PCA
**B.Tech CSE Mini Project | Intermediate Level**

**Team Members**
- Yashraj Sahu
- Aditya Malviya
- Riya Mane
- Saish Uplenchwar

## Objective
Predict the final Mathematics grade (`G3`, on a 0–20 scale) from student demographic, family, school, lifestyle and previous-grade information. The workflow uses preprocessing, Principal Component Analysis (PCA), and Linear Regression.

> Dataset note: This project uses the Kaggle Student Performance dataset (`student-mat.csv`). Download it from [Kaggle – Student Alcohol Consumption](https://www.kaggle.com/datasets/uciml/student-alcohol-consumption). Extract the ZIP and place `student-mat.csv` in the same folder as this notebook. The CSV uses a semicolon (`;`) separator.

**Important interpretation:** The input data includes `G1` and `G2`, which are earlier-period grades. Therefore, this model estimates final grade after earlier grades are available; it is not an early-term prediction made before any assessments.

## 1. Install packages (run once if needed)

In [ ]:
%pip install -q pandas numpy matplotlib scikit-learn joblib

## 2. Import libraries and load the dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

DATA_FILE = "student-mat.csv"

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        "student-mat.csv was not found. Download it from Kaggle, extract the ZIP, "
        "and place student-mat.csv in the same folder as this notebook."
    )

df = pd.read_csv(DATA_FILE, sep=";")

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
display(df.head())

## 3. Explore and validate the data

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nMissing values in the dataset:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

print("\\nFinal grade (G3) summary:")
display(df["G3"].describe())

print("\\nDataset information:")
df.info()

## 4. Prepare features and target

In [ ]:
# Target: final Mathematics grade.
# Features: all remaining columns, including G1 and G2 (previous-period grades).
X = df.drop(columns=["G3"])
y = df["G3"]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

# Standardize numerical columns and one-hot encode categorical columns.
# Dense output is required by the PCA implementation used below.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_features)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("Number of original input columns:", X.shape[1])
print("Numerical input columns:", len(numeric_features))
print("Categorical input columns:", len(categorical_features))

## 5. Split data, apply PCA, and train Linear Regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("pca", PCA(n_components=0.95, svd_solver="full")),
    ("regression", LinearRegression())
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Model training completed.")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("PCA components retained:", model.named_steps["pca"].n_components_)
print("Transformed feature count before PCA:",
      model.named_steps["preprocessing"].transform(X_train).shape[1])
print("Explained variance retained:",
      round(model.named_steps["pca"].explained_variance_ratio_.sum(), 4))

## 6. Evaluate the model

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MODEL PERFORMANCE")
print("-----------------")
print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")

print("\\nInterpretation:")
print(f"On average, predictions differ from actual grades by about {mae:.2f} grade points (MAE).")
print("R² measures the proportion of test-set grade variation explained by the model.")

## 7. Visualize actual and predicted grades

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.75)
plt.plot([0, 20], [0, 20], linestyle="--")
plt.xlabel("Actual final grade (G3)")
plt.ylabel("Predicted final grade")
plt.title("Actual vs Predicted Student Grades")
plt.xlim(0, 20)
plt.ylim(0, 20)
plt.grid(True, alpha=0.3)
plt.show()

## 8. View PCA explained variance

In [ ]:
pca = model.named_steps["pca"]
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
plt.axhline(0.95, linestyle="--", label="95% variance threshold")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA – Cumulative Explained Variance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 9. Predict the grade for a selected dataset student

In [ ]:
# Select a row by its zero-based position (0 to len(df)-1).
# This demonstrates prediction for a record from the dataset.
row_number = 0
student_record = X.iloc[[row_number]]
actual_grade = y.iloc[row_number]
predicted_grade = model.predict(student_record)[0]

print("Selected dataset row:", row_number)
print("Actual final grade (G3):", actual_grade)
print("Predicted final grade:", round(float(predicted_grade), 2))

## 10. Predict using a new student profile (optional template)

In [ ]:
# Edit the values in this dictionary to match the dataset's column names and categories.
# This example starts from an existing record so every required feature is present.
new_student = X.iloc[[0]].copy()

# Example edits (change these values if desired):
new_student.loc[:, "studytime"] = 3
new_student.loc[:, "absences"] = 2
new_student.loc[:, "G1"] = 12
new_student.loc[:, "G2"] = 13

new_prediction = model.predict(new_student)[0]
print("Estimated final grade (G3):", round(float(new_prediction), 2), "/ 20")

## 11. Save the trained model

In [ ]:
joblib.dump(model, "student_performance_model.pkl")
print("Model saved as student_performance_model.pkl")

## 12. Optional: simple prediction menu
Run this cell after training. Enter a row number between 0 and 394 to predict for an existing dataset record. Press `q` to quit.

In [ ]:
while True:
    choice = input("Enter student row number (0-394) or q to quit: ").strip()
    if choice.lower() == "q":
        print("Program closed.")
        break

    if not choice.isdigit():
        print("Please enter a valid whole number.")
        continue

    index = int(choice)
    if not 0 <= index < len(X):
        print(f"Enter a number from 0 to {len(X)-1}.")
        continue

    prediction = model.predict(X.iloc[[index]])[0]
    print(f"Predicted final grade for row {index}: {prediction:.2f} / 20")

## 13. Conclusion
This mini project demonstrates an end-to-end machine-learning workflow: loading a real Kaggle dataset, exploring and preprocessing mixed numerical/categorical data, reducing dimensionality with PCA, training a Linear Regression model, evaluating predictions, visualizing results, and saving the trained model.

The MAE, RMSE, R² score, PCA component count, and plotted results are generated from the downloaded CSV when the notebook is executed. Do not copy invented or placeholder metric values into the final report; use the values printed by your own notebook run.

## References
1. Kaggle: [Student Alcohol Consumption / Student Performance dataset](https://www.kaggle.com/datasets/uciml/student-alcohol-consumption)
2. Cortez, P. and Silva, A. (2008), Using Data Mining to Predict Secondary School Student Performance.
3. Scikit-learn documentation: [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html), and [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).